# Hardware-Aware Gateway. Kaggle bootstrap

Kaggle's free GPU quota is separate from Colab's and considerably larger,
30 hours a week against a handful of sessions. It is also the quieter
machine: the same end-to-end benchmark measured a standard deviation of 0.53
tok/s here against 3.68 on Colab, which is the difference between an effect
that resolves and one that does not.

**Before running**, open the right sidebar and set:

* **Accelerator**: GPU T4 x2
* **Internet**: On. Without it the clone, the install and the model download
  all fail. If the toggle will not move, Kaggle wants your phone verified
  under Settings first.


In [ ]:
import torch
assert torch.cuda.is_available(), (
    'No GPU. Right sidebar > Accelerator > GPU T4 x2, then rerun.'
)
print(torch.cuda.get_device_name(0))
print('torch ', torch.__version__)


## Run everything

Clone, install, correctness, op sweep, profile, end to end, CUDA graphs,
report. Under `set -e`, so the first failure stops it.

The one exception is the CUDA graph step, which is allowed to fail without
taking the run down and prints a banner if it does. Everything before it is
established; a failure there should not cost the results that already
succeeded.


In [ ]:
!curl -sL https://raw.githubusercontent.com/Rahu378/hardware-aware-gateway/main/scripts/colab_run.sh | bash


## Download the archive

Only if the cell above reached `Archive:`. This also copies it into
`/kaggle/working` so it shows up in the Output panel.


In [ ]:
import os, shutil

for src in ('/kaggle/working/artifacts.zip', '/content/artifacts.zip'):
    if os.path.exists(src):
        if src != '/kaggle/working/artifacts.zip':
            shutil.copy(src, '/kaggle/working/artifacts.zip')
        break
else:
    raise SystemExit('No archive found; the run did not finish.')

os.chdir('/kaggle/working')
from IPython.display import FileLink
FileLink('artifacts.zip')


---

Results land in `results/` and `profiles/`. Commit them from your machine so
the repository history carries the runs and not only the code.

`ncu` still needs a VM with performance-counter access; it fails on both
Kaggle and Colab with `ERR_NVGPUCTRPERM`. See `scripts/profile_ncu.sh`.
